# Model Bayesian Poisson untuk Gol Piala Dunia

Notebook ini menggunakan Bahasa Indonesia untuk mendukung presentasi. Fokus utamanya adalah menjelaskan inferensi Bayesian untuk data hitungan dalam konteks mata kuliah Mathematical and Statistical Foundations.

Aplikasi sepak bolanya adalah total gol per pertandingan Piala Dunia. Kita memakai Piala Dunia era modern yang sudah selesai untuk mengestimasi rata-rata gol, lalu mengecek apakah jumlah gol pada Piala Dunia 2022 konsisten dengan model.

## Model

Misalkan `Y` adalah total gol dalam satu pertandingan.

$$Y_i \mid \lambda \sim Poisson(\lambda)$$

Parameter `lambda` adalah rata-rata gol yang diharapkan per pertandingan.

Dalam pendekatan Bayesian, kita menetapkan prior Gamma untuk `lambda`:

$$\lambda \sim Gamma(\alpha, \beta)$$

Dengan parameterisasi rate, rata-rata prior adalah:

$$E[\lambda] = \frac{\alpha}{\beta}$$

## Update Posterior

Prior Gamma bersifat konjugat terhadap likelihood Poisson. Jika kita mengamati `n` pertandingan dengan total gol `sum_y`, maka:

$$\lambda \mid data \sim Gamma(\alpha + \sum y_i, \beta + n)$$

Ini adalah hasil matematis utama yang dipakai dalam notebook.

In [ ]:
from pathlib import Path
import json
import math
import statistics

ROOT = Path.cwd()
DATA_DIR = ROOT / "worldcup.json"

def load_worldcup_matches(year):
    path = DATA_DIR / str(year) / "worldcup.json"
    with path.open(encoding="utf-8") as file:
        data = json.load(file)
    return data["matches"]

def scored_full_time_matches(year):
    matches = load_worldcup_matches(year)
    return [match for match in matches if "score" in match and "ft" in match["score"]]

def total_goals_by_match(year):
    return [sum(match["score"]["ft"]) for match in scored_full_time_matches(year)]

def summarize_gol(years):
    rows = []
    for year in years:
        gol = total_goals_by_match(year)
        rows.append({
            "year": year,
            "matches": len(gol),
            "total_goals": sum(gol),
            "avg_gol": sum(gol) / len(gol),
        })
    return rows

def frequency_table(values):
    counts = {}
    for value in values:
        counts[value] = counts.get(value, 0) + 1
    return dict(sorted(counts.items()))

def ascii_bar_chart(rows, label_key, value_key, width=32):
    max_value = max(row[value_key] for row in rows) if rows else 0
    for row in rows:
        value = row[value_key]
        bar_length = round((value / max_value) * width) if max_value else 0
        bar = "#" * bar_length
        print(f"{row[label_key]!s:>6} | {bar:<{width}} {value:.3f}")

def credible_interval(probability_rows, mass=0.90):
    lower_tail = (1 - mass) / 2
    upper_tail = 1 - lower_tail
    cumulative = 0.0
    lower = None
    upper = None
    for row in probability_rows:
        cumulative += row["probability"]
        if lower is None and cumulative >= lower_tail:
            lower = row["gol"]
        if upper is None and cumulative >= upper_tail:
            upper = row["gol"]
            break
    return lower, upper

## Memuat dan Memeriksa Data

Model dibangun menggunakan Piala Dunia 2010, 2014, dan 2018 sebagai data latih. Piala Dunia 2022 dipakai sebagai data evaluasi.

In [ ]:
train_years = [2010, 2014, 2018]
test_year = 2022

summary_rows = summarize_gol(train_years + [test_year])
for row in summary_rows:
    print(
        f"{row['year']}: {row['matches']} pertandingan, "
        f"{row['total_goals']} gol, "
        f"rata-rata = {row['avg_gol']:.3f}"
    )

print("\nRata-rata gol per turnamen:")
ascii_bar_chart(summary_rows, "year", "avg_gol")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 4))
    plt.bar([str(row["year"]) for row in summary_rows], [row["avg_gol"] for row in summary_rows])
    plt.axhline(2.5, color="tab:red", linestyle="--", label="Prior mean = 2.5")
    plt.title("Rata-rata Gol per Pertandingan Piala Dunia")
    plt.xlabel("Turnamen")
    plt.ylabel("Rata-rata gol per pertandingan")
    plt.legend()
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib tidak tersedia; grafik teks ditampilkan sebagai pengganti.")

## Pilihan Prior

Kita memakai prior lemah dengan pusat sekitar 2.5 gol per pertandingan:

$$\lambda \sim Gamma(5, 2)$$

Rata-rata prior adalah `5 / 2 = 2.5`. Nilai ini masuk akal untuk Piala Dunia era modern, tetapi pengaruhnya tidak terlalu kuat dibandingkan 192 pertandingan latih.

In [ ]:
alpha_prior = 5.0
beta_prior = 2.0

train_goals = []
for year in train_years:
    train_goals.extend(total_goals_by_match(year))

train_total_goals = sum(train_goals)
train_match_count = len(train_goals)

alpha_post = alpha_prior + train_total_goals
beta_post = beta_prior + train_match_count

prior_mean = alpha_prior / beta_prior
posterior_mean = alpha_post / beta_post

print(f"Jumlah pertandingan latih: {train_match_count}")
print(f"Total gol data latih: {train_total_goals}")
print(f"Rata-rata prior lambda: {prior_mean:.3f}")
print(f"Alpha posterior: {alpha_post:.3f}")
print(f"Beta posterior: {beta_post:.3f}")
print(f"Rata-rata posterior lambda: {posterior_mean:.3f}")

print("\nPenjelasan prior:")
print("Gamma(5, 2) memiliki rata-rata 2.5, yaitu keyakinan awal yang lemah dan masuk akal untuk gol Piala Dunia modern.")
print("Setelah 192 pertandingan latih, posterior lebih banyak dipengaruhi data observasi daripada prior.")

try:
    import matplotlib.pyplot as plt
    import numpy as np

    def gamma_pdf(x, alpha, beta):
        log_densitas = alpha * math.log(beta) - math.lgamma(alpha) + (alpha - 1) * math.log(x) - beta * x
        return math.exp(log_densitas)

    xs = np.linspace(0.01, 5, 300)
    prior_densitas = [gamma_pdf(float(x), alpha_prior, beta_prior) for x in xs]
    posterior_densitas = [gamma_pdf(float(x), alpha_post, beta_post) for x in xs]

    plt.figure(figsize=(7, 4))
    plt.plot(xs, prior_densitas, label="Prior Gamma(5, 2)")
    plt.plot(xs, posterior_densitas, label="Posterior setelah 2010-2018")
    plt.title("Keyakinan Prior vs Posterior terhadap Lambda")
    plt.xlabel("lambda: ekspektasi gol per pertandingan")
    plt.ylabel("densitas")
    plt.legend()
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib/numpy tidak tersedia; rata-rata prior dan posterior ditampilkan di atas.")

## Distribusi Prediktif posterior

Distribusi prediktif posterior menjawab pertanyaan:

> Berapa banyak gol yang mungkin terjadi pada pertandingan berikutnya, setelah model belajar dari data latih?

Untuk model Gamma-Poisson, distribusi prediktif berbentuk negative binomial. Fungsi di bawah menghitung probabilitas bahwa satu pertandingan baru memiliki tepat `k` gol.

In [ ]:
def negative_binomial_predictive_pmf(k, alpha, beta):
    """Posterior predictive P(Y_new = k) for Poisson-Gamma model.

    Menggunakan Gamma(shape=alpha, rate=beta). Untuk satu pertandingan baru,
    P(Y=k) = C(k+alpha-1, k) * (beta/(beta+1))^alpha * (1/(beta+1))^k.
    Bentuk log-gamma dipakai agar perhitungan stabil untuk nilai alpha non-integer.
    """
    log_coef = math.lgamma(k + alpha) - math.lgamma(alpha) - math.lgamma(k + 1)
    log_prob = log_coef + alpha * math.log(beta / (beta + 1)) + k * math.log(1 / (beta + 1))
    return math.exp(log_prob)

predictive_probs = {
    gol: negative_binomial_predictive_pmf(gol, alpha_post, beta_post)
    for gol in range(0, 9)
}

for gol, probability in predictive_probs.items():
    print(f"P(pertandingan berikutnya memiliki {gol} gol) = {probability:.4f}")

print(f"P(pertandingan berikutnya memiliki 9+ gol) = {1 - sum(predictive_probs.values()):.4f}")

predictive_probability_rows = [
    {"gol": gol, "probability": probability}
    for gol, probability in predictive_probs.items()
]
interval_90 = credible_interval(predictive_probability_rows, mass=0.90)
print(f"Interval kredibel prediktif posterior 90% untuk satu pertandingan: {interval_90[0]} sampai {interval_90[1]} gol")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 4))
    plt.bar(list(predictive_probs.keys()), list(predictive_probs.values()))
    plt.title("Distribusi Prediktif posterior")
    plt.xlabel("Total gol pada pertandingan berikutnya")
    plt.ylabel("Probabilitas prediksi")
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib tidak tersedia; probabilitas prediksi ditampilkan di atas.")

## Evaluasi pada 2022

Model sederhana ini memberikan distribusi prediktif posterior yang sama untuk setiap pertandingan masa depan karena model belum mengetahui kekuatan tim, kekuatan lawan, atau fase turnamen. Ini adalah keterbatasan, tetapi membuat proses update Bayesian lebih mudah dijelaskan.

In [ ]:
test_goals = total_goals_by_match(test_year)
test_average = sum(test_goals) / len(test_goals)

absolute_errors = [abs(goal_count - posterior_mean) for goal_count in test_goals]
squared_errors = [(goal_count - posterior_mean) ** 2 for goal_count in test_goals]

mae = sum(absolute_errors) / len(absolute_errors)
rmse = math.sqrt(sum(squared_errors) / len(squared_errors))

def poisson_pmf(k, lam):
    return math.exp(-lam + k * math.log(lam) - math.lgamma(k + 1))

def predictive_probability_for_observed(k):
    return negative_binomial_predictive_pmf(k, alpha_post, beta_post)

mean_log_predictive_probability = statistics.mean(
    math.log(predictive_probability_for_observed(goal_count))
    for goal_count in test_goals
)

actual_frequency = frequency_table(test_goals)
actual_vs_predicted_rows = []
for gol in range(0, max(max(actual_frequency), max(predictive_probs)) + 1):
    actual_count = actual_frequency.get(gol, 0)
    predicted_probability = negative_binomial_predictive_pmf(gol, alpha_post, beta_post)
    actual_vs_predicted_rows.append({
        "gol": gol,
        "actual_count": actual_count,
        "actual_share": actual_count / len(test_goals),
        "predicted_probability": predicted_probability,
    })

lower_90, upper_90 = credible_interval(
    [{"gol": row["gol"], "probability": row["predicted_probability"]} for row in actual_vs_predicted_rows],
    mass=0.90,
)
coverage_90 = sum(lower_90 <= goal_count <= upper_90 for goal_count in test_goals) / len(test_goals)

print(f"Jumlah pertandingan 2022: {len(test_goals)}")
print(f"Rata-rata gol 2022: {test_average:.3f}")
print(f"Rata-rata prediktif posterior: {posterior_mean:.3f}")
print(f"Mean absolute error: {mae:.3f}")
print(f"Root mean squared error: {rmse:.3f}")
print(f"Rata-rata log probabilitas prediksi: {mean_log_predictive_probability:.3f}")
print(f"Interval prediksi 90%: {lower_90} sampai {upper_90} gol")
print(f"Coverage empiris 2022 untuk interval tersebut: {coverage_90:.3f}")

print("\nFrekuensi aktual 2022 vs probabilitas prediksi:")
print("gol | jumlah aktual | proporsi aktual | probabilitas prediksi")
for row in actual_vs_predicted_rows:
    print(
        f"{row['gol']:>5} | {row['actual_count']:>12} | "
        f"{row['actual_share']:.3f}        | {row['predicted_probability']:.3f}"
    )

try:
    import matplotlib.pyplot as plt
    x = [row["gol"] for row in actual_vs_predicted_rows]
    actual = [row["actual_share"] for row in actual_vs_predicted_rows]
    predicted = [row["predicted_probability"] for row in actual_vs_predicted_rows]

    plt.figure(figsize=(8, 4))
    plt.bar([value - 0.2 for value in x], actual, width=0.4, label="Proporsi aktual 2022")
    plt.bar([value + 0.2 for value in x], predicted, width=0.4, label="Probabilitas prediksi")
    plt.title("Jumlah Gol Aktual 2022 vs Model Prediktif Posterior")
    plt.xlabel("Total gol in match")
    plt.ylabel("Proporsi / probabilitas")
    plt.legend()
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib tidak tersedia; tabel perbandingan ditampilkan di atas.")

## Klasifikasi Opsional: Lebih dari 2.5 Gol

Model Bayesian Poisson adalah model hitungan, bukan model klasifikasi. Namun, kita dapat membuat tugas klasifikasi sekunder:

- positif: pertandingan memiliki lebih dari 2.5 gol, artinya 3 gol atau lebih
- negatif: pertandingan memiliki 2 gol atau kurang

Di bagian inilah true positive, false positive, true negative, dan false negative menjadi bermakna.

In [ ]:
prob_over_2_5 = 1 - sum(
    negative_binomial_predictive_pmf(k, alpha_post, beta_post)
    for k in range(0, 3)
)
predict_over_2_5 = prob_over_2_5 >= 0.5

tp = fp = tn = fn = 0
for actual_goals in test_goals:
    actual_over_2_5 = actual_goals >= 3
    if predict_over_2_5 and actual_over_2_5:
        tp += 1
    elif predict_over_2_5 and not actual_over_2_5:
        fp += 1
    elif not predict_over_2_5 and not actual_over_2_5:
        tn += 1
    else:
        fn += 1

print(f"P(lebih dari 2.5 gol) = {prob_over_2_5:.3f}")
print(f"Ambang keputusan: prediksi lebih dari 2.5 gol jika probabilitas >= 0.5")
print({"TP": tp, "FP": fp, "TN": tn, "FN": fn})

## Perbandingan Opsional: Semua Piala Dunia Sebelumnya

Satu poin diskusi yang berguna adalah apakah turnamen lama perlu ikut dipakai. Piala Dunia awal memiliki format berbeda dan sering memiliki rata-rata gol lebih tinggi. Kode di bawah membandingkan model yang dilatih dengan semua turnamen sebelum 2022 terhadap model berbasis data modern.

In [ ]:
all_prior_years = [
    1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970,
    1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006,
    2010, 2014, 2018,
]

all_prior_goals = []
for year in all_prior_years:
    all_prior_goals.extend(total_goals_by_match(year))

alpha_all = alpha_prior + sum(all_prior_goals)
beta_all = beta_prior + len(all_prior_goals)
posterior_mean_all = alpha_all / beta_all
comparison_rows = [
    {"model": "2010-2018", "mean": posterior_mean},
    {"model": "1930-2018", "mean": posterior_mean_all},
    {"model": "Aktual 2022", "mean": test_average},
]

print(f"Rata-rata posterior data modern: {posterior_mean:.3f}")
print(f"Rata-rata posterior semua tahun sebelum 2022: {posterior_mean_all:.3f}")
print(f"Rata-rata aktual 2022: {test_average:.3f}")
print("\nPerbandingan ekspektasi/aktual gol per pertandingan:")
ascii_bar_chart(comparison_rows, "model", "mean")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 4))
    plt.bar([row["model"] for row in comparison_rows], [row["mean"] for row in comparison_rows])
    plt.title("Pilihan Data Latih Modern vs Historis")
    plt.ylabel("Gol per pertandingan")
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib tidak tersedia; grafik teks perbandingan ditampilkan di atas.")

## Interpretasi

Notebook ini menunjukkan alur Bayesian secara lengkap:

1. Memilih model probabilitas untuk data hitungan.
2. Memilih prior untuk laju rata-rata yang belum diketahui.
3. Memperbarui prior dengan data gol Piala Dunia.
4. Menggunakan distribusi prediktif posterior untuk menalar pertandingan yang ditahan sebagai evaluasi.
5. Menjelaskan keterbatasan model secara jujur.

Hasilnya bukan model prediksi sepak bola yang lengkap. Hasilnya adalah model statistik yang jelas untuk menjelaskan inferensi Bayesian Poisson.